## Scraping House Journal Data

The MA House Journal API as defined on the [MA Legislature Website](https://malegislature.gov/api/swagger/index.html?url=/api/swagger/v1/swagger.json#/RollCalls/RollCalls_GetRollCallAll) just does not work. Even the following example curl request fails to locate a journal that clearly exists:

```console
curl -X GET "https://malegislature.gov/api/Documents/hj09172025/RollCalls" -H "accept: application/json"
```

As a result, in order to properly automate scraping the pdf files, we can do the following approach.

A download of a PDF file has the following URL:

https://malegislature.gov/Journal/House/194/RollCalls/hj09172025_RollCalls.pdf

The two most important paramters are "194" and "hj09172025". 194 is the General Court Number and "hj09172025" is the document number. A document number is broken into 2 components

hj - House Journal
09172025 - The date the journal was published.

As a result, we can programmatically check for each valid URL and download a PDF for it. This unfortunately runs into the problem of doing on average # of total General Court sessions * 365 requests per year.

This could be handled by simply manually checking for each link, copying it and downloading through Python. However, for the sake of enrichment and learning, the following code automates getting every possible valid House Journal Roll call vote.

In [34]:
import requests
import pandas as pd
import time
import datetime
import random

In [26]:
# Create arrays for each General Court Numbers
court_numbers = [i for i in range(183, 195)]
court_years = [i for i in range(2003, 2026, 2)]
months = [i for i in range(1, 13)]
days = [i for i in range(1, 32)]

In [37]:
# Create a base url to modify
base = 'https://malegislature.gov/Journal/House/'

# Define a chunk_size for reading pdfs
chunk_size = 2000

In [41]:
def scrape_house(use_delay = False):
    # Loop through each court number
    for i, court_number in enumerate(court_numbers):

        start_year = court_years[i]
        end_year = start_year + 2

        while start_year != end_year and start_year != 2026:
            for month in months:
                for day in days:

                    # Create a url to request
                    date_str = f'hj{month:02d}{day:02d}{start_year}'
                    url = f'{base}{court_number}/RollCalls/{date_str}_RollCalls.pdf'

                    print(f"[{date_str[2:]}] - Searching for RollCall Vote pdf...")

                    # Add random delay to request
                    if use_delay:
                        time.sleep(random.uniform(1, 3))

                    r = requests.get(url, stream=True)

                    if r.status_code == 200:
                        # Open pdf file to write output to
                        path = f'data/HJRollCalls/{date_str}.pdf'
                        with open(path, 'wb') as f:
                            for chunk in r.iter_content(chunk_size=chunk_size):
                                f.write(chunk)

                        print("Found Roll Call and wrote to path: ", path)
                    else:
                        print("No roll call for this date!")

            #Increment start year
            start_year += 1
            return


In [42]:
scrape_house()

[01012003] - Searching for RollCall Vote pdf...
No roll call for this date!
[01022003] - Searching for RollCall Vote pdf...
No roll call for this date!
[01032003] - Searching for RollCall Vote pdf...
No roll call for this date!
[01042003] - Searching for RollCall Vote pdf...
No roll call for this date!
[01052003] - Searching for RollCall Vote pdf...
No roll call for this date!
[01062003] - Searching for RollCall Vote pdf...
No roll call for this date!
[01072003] - Searching for RollCall Vote pdf...
No roll call for this date!
[01082003] - Searching for RollCall Vote pdf...
Found Roll Call and wrote to path:  data/HJRollCalls/hj01082003.pdf
[01092003] - Searching for RollCall Vote pdf...
Found Roll Call and wrote to path:  data/HJRollCalls/hj01092003.pdf
[01102003] - Searching for RollCall Vote pdf...
No roll call for this date!
[01112003] - Searching for RollCall Vote pdf...
No roll call for this date!
[01122003] - Searching for RollCall Vote pdf...
No roll call for this date!
[0113200

## Attempt to parse a House Roll Call PDF

### Combine using table operations

In [43]:
import pymupdf

In [49]:
doc = pymupdf.open("data/HJRollCalls/hj01082003.pdf")
for page in doc:
    text = page.get_text().splitlines()
    print(text)

#Outputs individual values from PDF, which I think would be the most useful/easy to manipulate as needed

['                         MASSACHUSETTS HOUSE OF REPRESENTATIVES                       ', '  ', '                                         QUORUM                                       ', '  ', '  Yea and Nay                                                         01/08/03 01:44 P', '   No.    1                     151 YEAS   0 NAYS   9 N/V                             ', '  ', '  ', '  ', '  Y    Mr. Speaker     Y    Fresolo         Y    Murphy, K.J.    Y    Swan            ', '  Y    Asselin         Y    Galvin          Y    Nangle          Y    Teahan          ', '  Y    Atkins          Y    Garry           Y    Naughton        Y    Timilty         ', '  Y    Atsalis           X  Gobi            Y    Nyman           Y    Tobin           ', "  Y    Ayers           Y    Goguen          Y    O'Brien         Y    Toomey          ", "  Y    Balser          Y    Golden B.P.     Y    O'Flaherty      Y    Torrisi         ", '  Y    Binienda        Y    Golden T.A.     Y    Owens-Hicks     Y  

In [48]:
#Referenced https://artifex.com/blog/extracting-tables-from-pdfs-with-pymupdf
page1 = doc[0]
tables = page1.find_tables(strategy = "text")
table1 = tables[0]
df = table1.to_pandas()

df.tail() #Need to reshape/restructure to be just two columns, name and vote


,Yea a,nd Nay,Col2,Col3,Col4,Col5,Col6,01/08/03 01
37,Y,Fennell,Y,Mariano,Y,St. Fleur,Y,Poirier
38,Y,Festa,Y,Marzilli,Y,Story,Y,Polito
39,Y,Finegold,X,Miceli,X,Straus,Y,Pope
40,Y,Flynn,Y,"Murphy, C.A.",Y,"Sullivan, D.B.",Y,Webster
41,Y,Fox,Y,"Murphy, J.M.",Y,"Sullivan, J.C.",Y,Lantigua
